# Deep Network Developments, Coding test - Variant B (12 May 2026)

**In the Coding test, you need to train and evaluate a simple Multi-layer Perceptron (MLP) neural network on a multi-class classification task using the PyTorch library.**

The Coding test consists of several subtasks (A - G) that build on one another. The assignment includes an automatic testing script that checks the correctness of your solutions for the individual subtasks. Therefore, **do not modify the structure of the notebook**, do not split it into separate parts, and do not alter the already written sections; only enter your solutions in the designated places, otherwise we will not be able to evaluate your work. Your solutions must be written between the `# Implement your solution BELOW` and `# Implement your solution ABOVE` lines. If the message "Tester: .... OK" appears when running a solved subtask, the testing script did not find an error in your solution. However, the tester is not exhaustive, so it may fail to detect certain kinds of mistakes.

**Classification accuracy achievable in this task:** With a correct solution, approximately 60-70% (multi-class) accuracy can be achieved. (The label categories will be roughly balanced.) If the best result your neural network can achieve on the test set is significantly worse or significantly better than this, that may indicate an error and you should try to identify it. For example, you can compare the results of the subtasks with the task descriptions and with the original CSV dataset files.

**Submitting the completed assignment:** Only the notebook file (`.ipynb`) containing your final solution must be uploaded to Canvas, on the submission page for the "Coding test (12 May 2026)" assignemnt. No other files should be uploaded.  
**Please do not modify the notebook structure and do not split it into separate tasks.**

**Available time:** 180 minutes

**Grading:** A maximum of 30 points can be earned. See the points available for each individual subtask in the task descriptions. You must earn at least 12 points on this test to be able to pass the course. If this is not achieved, a retake will be required.

**Constraint:** Whenever possible, we expect efficient, vectorized solutions. A loop with a large number of iterations (50+) or similarly inefficient constructs in the code may result in point deductions. The following do **not** count as vectorized solutions: recursion, list/set/dict comprehensions, iterators, generators, `map()`, `np.apply_along_axis()`, `np.vectorize()`, `np.frompyfunc()`, and similar approaches.

Exceptions where longer loops may be used without penalty:
- In task A).
- During network training, inside the training/validation/test loop, whether iterating over epochs or batches.


---

Download the testing script and import the required modules.


In [1]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from dndmsc26spring_zh1b_tester import Tester

## **A**: Loading the dataset - 1 point

**Information about the datasets:**  
We will work with a synthetic dataset (`orangejuice_dnd26spring_zh_b.csv`) that contain the chemical properties of different types of orange juice, as well as the subjective tasting-based ratings (1-9) given by three orange-juice experts. Each row of a dataset contains the data for one type of orange juice, while the columns describe the individual chemical properties. The last three columns (named `score1`, `score2`, `score3` in the header) contain the scores given by the three experts.

In the Coding test, we will first **load the dataset containing orange juice data**, **identify and handle missing values**, **split the dataset into training/validation and test splits**m then **define three quality categories ("lower-shelf" / "middle-shelf" / "top-shelf" orange juices), assign all of the juices into one of these three categories based on the experts' scores, and then learn to predict this category label from the chemical properties of the juices using a neural network.**

The testing script loads the orange juice dataset stored in text form and places its content into the `content` string. Below, we print the length and the first 500 characters of this string. Each row of the dataset file specifies the values of the variables for one data point (one orange juice). Within a row, the values corresponding to the variables are **separated by commas**. The first row of each dataset string contains the variable names.

Your task is to **convert the `content` string into a NumPy array containing the values of the dataset variables.** The data type of the array should be `np.float32` (floating point). Place the variable values in the `dataset_noisy` array with shape `(n_orangejuice, n_variables)`. The input variables and labels will be separated later.

The dataset contains missing values in some places: some of the data was not available when the dataset was recorded. More specifically, missing values can be found in the columns containing measurements related to the amount of dissolved solid material and the pH of the orange juices. These are the columns with index #0 ("dissolved solids") and #1 ("pH"). The missing values have been replaced by zeros (`0`). In this subtask, you do not need to handle these values separately yet; you may read them as numerical values equal to 0, together with the other values.


In [2]:
tester = Tester()
content = tester.get_dataset_content()

print("Number of characters in dataset:", len(content))
print(content[:500])


Number of characters in dataset: 197722
"dissolved solids","pH","titratable acidity","electrical conductivity","flavonoids","aldehydes","alcohols","esters","ketones","terpenoids","hydrocarbons","score1","score2","score3"
17.171,3.922,7.120,1.64,126.042,0.907,1.175,1.158,0.002,33.098,45.674,1,1,2
0,3.234,10.39,1.752,101.7,0.985,2.221,0.298,0.013,13.306,0.,5,1,5
12.224,3.241,9.156,2.126,192.418,0.627,0.713,0.668,0.055,0.,24.219,6,6,6
10.404,3.542,8.644,1.592,246.394,0.898,2.267,0.156,0.070,8.097,0.,8,9,8
8.329,3.668,8.321,2.067,211.514,


In [14]:
# implement your solution BELOW
import io
dataset_noisy  = np.genfromtxt(
    io.StringIO(content),
    delimiter = ',',
    skip_header=1,
    dtype=np.float32
)
print(dataset_noisy.shape)

# implement your solution ABOVE

tester.test('dataset_load', dataset_noisy)

(2697, 14)
Tester: Dataset loading OK


## **B**: Handling missing data - up to 5 points

Fortunately, the missing values are represented by numbers (zeros), so we would not strictly have to deal with them. However, if we leave them in the array as values equal to 0, our neural network may interpret them as meaning that the amount of dissolved solids or the pH value of our orange juice in question is exactly zero, which is obviously not true. As a result, the network may draw biased conclusions. **Solve the task described in one of the options below!**

### **Option 1 (easiest)** - 0 points

In this option, we do not handle the missing data: the missing values are kept unchanged as values equal to 0. Thus, you do not need to change anything in the dataset. Simply assign the `dataset_noisy` array to the variable named `dataset`.

### **Option 2 (not very difficult)** - 3 points

**Create two additional variables that indicate whether a valid value is present in the variables with index #0 and #1 for the given data points (juice), or not.** If a valid value is present, the new variable should take the value 1. If the value is missing (0), the new variable should take the value 0 as well. Place the original data and the new variables in the `dataset` array: the columns of the new variables should be inserted, in the appropriate order, **after** the columns containing the first two variables.

Since only the columns (variables) with index #0 and #1 of our 14-variable dataset can contain missing values, the new `dataset` array will contain 16 variables, and the two new variables will be placed in columns with index #2 and #3. For example, in column #2, a given row should contain a zero if column #0 has a missing value (0) in that row; otherwise, it should contain a one. Column #3 should be constructed similarly, based on column #1. Further, previously existing columns are pushed back accordingly (e.g., original column #2 will become column #4, original column #3 will become column #5, etc.)

You may handle the two variables/columns in separate lines of code. (Read the next option as well before starting your solution.)

### **Option 3 (a bit more difficult)** - 5 points

Implement the task described in Option 2, but solve it in a fully vectorized way: handle the two columns containing missing values at the same time, using shared operations.


In [36]:
# implement your solution BELOW

valid_col0 = (dataset_noisy[:, 0] !=  0).astype(np.float32).reshape(-1, 1)
valid_col1 = (dataset_noisy[:, 1] != 0).astype(np.float32).reshape(-1, 1)

dataset = np.concatenate(
    [dataset_noisy[:, :2], valid_col0, valid_col1, dataset_noisy[:, 2:]],
    axis=1
).astype(np.float32)

print(dataset.shape)


# implement your solution ABOVE

tester.test('dataset_fill_missing', dataset)

(2697, 16)
Tester: Dataset fill missing: Identified solution aiming for option#2 or #3...
Tester: Dataset fill missing: Option #2 or #3 OK


## **C**: Splitting into training, validation, and test sets - 3 points

**Randomly shuffle the data points** in the `dataset` array. In general, this is advisable because the elements in a dataset may be ordered according to some of their properties. Without shuffling, the label distributions in the separated sets could differ substantially.

Then **split the shuffled array into training, validation, and test sets.** Place 600 data points (rows) in the validation set, 600 data points in the test set, and the remaining data points in the training set (the order does not matter). Name the three arrays containing the training, validation, and test sets `dataset_split_train`, `dataset_split_val`, and `dataset_split_test`, respectively.


In [2]:
# implement your solution BELOW

rng = np.random.default_rng(42)
indices = rng.permutation(dataset.shape[0])
dataset_shuffled = dataset[indices]


dataset_split_val = dataset_shuffled[:600].astype(np.float32)
dataset_split_test = dataset_shuffled[600:1200].astype(np.float32)
dataset_split_train = dataset_shuffled[1200:].astype(np.float32)

print(dataset_split_train.shape, dataset_split_val.shape, dataset_split_test.shape)

# implement your solution ABOVE

tester.test('dataset_split', dataset_split_train, dataset_split_val, dataset_split_test)

NameError: name 'np' is not defined

## **D**: Creating data iterators for the multi-class classification task - 6 points

Gradient-based training of neural networks is most often implemented using iterators that traverse our dataset and produce the input and label batches needed for training the neural networks.

Similarly to the homework assignment, this time we will also train our neural network using iterators. In this subtask, you must **create three iterable objects** named `dataloader_cl_train`, `dataloader_cl_val`, and `dataloader_cl_test`, which traverse the arrays containing the training, validation, and test sets created in subtask C. In the multi-class classification task, we use all variables in the dataset as input variables except for the scores given by the orange juice experts (`score1`, `score2`, `score3`, the last three columns in the dataset). As labels, we define three categories and assign juices from the dataset to these categories according to the following rules:

- **"Bottom-shelf orange juices"** category, category index #0: orange juices that were rated 2 or lower by at least one juice expert.
- **"Top-shelf orange juices"** category, category index #2: orange juices for which no juice expert gave a score lower than 5, and for which the average of the three  experts' scores is at least 6.
- **"Middle-shelf orange juices"** category, category index #1: orange juices that belong neither to the bottom-shelf category nor to the top-shelf category.

Accordingly, the two tensors returned by the iterators in each step must have the following shapes:
- The first tensor contains, for the data points (juices) assigned to the categories, the values of all the variables from the dataset except the scores, with data type `torch.float32`. Its shape is `(batch_size, 11)` if you solved option B/1 and `(batch_size, 13)` if you solved option B/2 or B/3.
- The second tensor (containing the labels) must contain the category of the orange juices (that is, values 0, 1 or 2, with data type `torch.int64`, in the form of category labels). The shape of the tensor is `(batch_size,)`. As before, `batch_size` denotes the number of samples in a batch and must be set manually (e.g., 32).

The implementation of the iterable objects can be done in several ways, either custom-written or by using the `Dataset` and `DataLoader` classes from the `torch.utils.data` module. Make sure that the **batches returned by the iterators are expected to contain a mixture of samples from all categories.**

For this task, also **print the category distribution of the different dataset splits.** In other words, show how many data points fall into categories #0, #1 and #2.
For example:
```
Training set -> category #0: 415, category #1: 604, category #2: 478
Validation set -> category #0: 167, category #1: 236, category #2: 197
Test set -> category #0: 163, category #1: 222, category #2: 215
```

(If you see significantly different distributions, there might be an error in your code.)

**Be careful: the tester script cannot fully verify the contents of the tensors returned by the iterators.**


In [1]:
# implement your solution BELOW

def sale_label(arr):
    scores = arr[:, -3:]
    bottom = np.any(scores <= 2, axis=1)
    top = (np.all(scores >= 5, axis=1)) & (np.mean(scores, axis=1) >= 6)
    labels = np.ones(arr.shape[0], dtype=np.int64) 
    labels[bottom] = 0
    labels[top] = 2
    return labels

class OrangeJuiceDataset(Dataset):
    def __init__(self, arr):
        self.X = torch.tensor(arr[:, 3], dtype=torch.float32)
        self.y = torch.tensor(sale_label(arr), dtype=torch.int64)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        x = torch.tensor(self.X[index], dtype=torch.float32)
        y = self.y
        return x, y

train_dataset = OrangeJuiceDataset(dataset_split_train)
val_dataset = OrangeJuiceDataset(dataset_split_val)
test_dataset = OrangeJuiceDataset(dataset_split_test)
        
batchsize = 32

dataloader_cl_train = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
dataloader_cl_val = DataLoader(val_dataset, batch_size=batchsize, shuffle=True)
dataloader_cl_test = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)
    
print(dataloader_cl_train, dataloader_cl_val, dataloader_cl_test)

# implement your solution ABOVE

tester.test('cl_iter', dataloader_cl_train, dataloader_cl_val, dataloader_cl_test)

NameError: name 'Dataset' is not defined

## **E**: Defining the neural network - 3 points

**Define the class that implements the neural network to be used for the multi-class classification task.** The class should be a subclass of the general `torch.nn.Module` class. Then instantiate it and assign the instance to the variable named `multiclass_cl_model`.

The neural network should **contain 2 fully connected layers**. The **first layer** should have 20 neurons, and you should apply a ReLU activation function to it. The number of neurons and the activation function of the **second layer** are determined by the multi-class classification task to be performed. During training, set individual elements of the vector representation obtained as the output of the ReLU activation function to zero with 10% probability. (That is, you will need a dropout "layer" after the ReLU.)

The new class must implement the `forward(self, x)` member function, which evaluates the neural network (hypothesis function) on the input tensor `x`. The `forward` function will receive the input tensors coming from the data iterator defined above and produce the label estimates from them. It is advisable to initialize the layers of the neural network in the constructor of the class.

**Reminder:** Make sure that dropout is active only during training. (This was covered in the lecture.)


In [ ]:
# implement your solution BELOW

class MultiClassModel(nn.Module):
    def __init__(self, n_input_cols):
        super().__init__()
        self.layer1 = nn.Linear(n_input_cols, 20)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.1)
        self.layer2 = nn.Linear(20, 3)
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.layer3(x)
        return x
multiclass_cl_model = MultiClassModel()
    

# implement your solution ABOVE

tester.test('cl_model_architecture', multiclass_cl_model)

AssertionError: Tester: Your 'multiclass_cl_model' neural network architecture does not follow the task description (incorrect parameter count).

## **F**: Training the network for the multi-class classification task - 6 points

- **Train the** `multiclass_cl_model` **neural network** on the training set using the `dataloader_cl_train` iterator, and use the `dataloader_cl_val` iterator for validation. A single pass through the training and validation datasets by the iterators (possibly in random order) defines one _epoch_.

  Use the **loss function** that is standard for multi-class classification during training.

  For training, choose a gradient-based **optimization algorithm** from the `torch.optim` module (for example, `SGD`, `Adam`, `RMSprop`, etc.), and also choose an appropriate **learning rate**.

- Training should be stopped by the **early stopping** technique. If the validation loss does not improve for a given number of epochs (`patience`), training should end, and **the weights of `multiclass_cl_model` should be restored from the epoch in which the validation loss was best.** This should happen automatically.

- **Measure the losses on the training and validation sets in every epoch, then after training has finished, plot how these values changed during training on a single shared graph.** You may use the `matplotlib` library, for example, to draw the plots. Make sure that the differences between the curves remain clearly visible at the end of training as well. If necessary, you may manually set which part of the y-axis the graph should show.

  **The graph should include a legend** indicating which curve describes the training loss and which one describes the validation loss.

- After training has finished, **measure the average loss on the full test set** and assign the resulting value to the variable `test_ce`.

- **Also compute the _accuracy_ metric on the samples of the test set** and assign the resulting value to the variable `test_acc`. The _accuracy_ metric gives the proportion of correctly classified data points. For example, if we estimate the categories of 150 samples and obtain the correct category in 75 cases, the value of _accuracy_ is 0.5. Make sure to compute it in a way that is compatible with multi-class classification.

- **Select a few samples from the test set** (for example, using the `dataloader_cl_test` iterator), **estimate their labels** from the values of the input variables, and then **print the predicted and true labels for the individual data points (juices) as categories** (for example: `Prediction: 2, true label: 1`). This lets us see, through examples, how good the estimates produced by our neural network are.

- Finally, **compute the confusion matrix of the network on the test set and plot it**! The confusion matrix is a k x k matrix for k categories, where the element at index `[i,j]` gives how many data points belonging to category `i` in the dataset were predicted by our model as category `j`. Good model performance is indicated when as many elements as possible fall on the main diagonal of the confusion matrix. For the computation, you may also use an external library, but it can be done in NumPy as well. For plotting, you may for example use the `imshow` function from `matplotlib.pyplot`. Make sure that it is clearly visible on the plot which axis lists the true categories and which lists the predicted categories, that the category indices are visible at the ends of the rows and columns, and that the cells contain the corresponding sample counts!


In [ ]:
# implement your solution BELOW






# implement your solution ABOVE

tester.test('cl_model_learning', test_ce, test_acc)

## **G**: Macro-averaged accuracy metric - 3 + 3 points

**!! Solve this task with the use of only Python, NumPy and/or PyTorch ! Do not use Scikit-learn or other libraries for this task !!**

The usual accuracy metric cannot handle imbalanced datasets properly. For example, if 80 percent of our data points belong to category 1, and our model learns only to estimate category label 1 in every case, regardless of the input, then the model achieves 80% accuracy even though it has not learned anything useful. Below, you must implement a balanced variant of the accuracy metric for multi-class cases with more than two categories.

For its calculation, we can start from the binary case (two categories). With two categories, we choose one category and call it the positive category, while the other category becomes the negative category. The binary accuracy metric is then the following:

$$ Binary\ accuracy = \dfrac{TP + TN}{TP + TN + FP + FN} = \dfrac{TP + TN}{\text{number of data points}}$$

where TP is the number of true positives (that is, cases where the positive category was estimated correctly), TN is the number of true negatives (that is, cases where the negative category was estimated correctly), FP is the number of false positives (cases where the data point was incorrectly assigned to the positive category), and FN is the number of false negatives (cases where the data point was incorrectly assigned to the negative category). Thus, the numerator is equal to the number of correctly classified elements in the binary task, and the quotient itself is the proportion of correctly classified elements in the binary task.

Since we now have three categories, we must extend the above metric to the multi-class classification case. In this case, we compute the above quotient three times, independently for the three categories, such that in each case the selected category is treated as the positive category and all other categories together form the negative category. **The macro-averaged accuracy is the average of the binary accuracy values computed for the three categories.**

### **Base task**: 3 points

**Compute the macro-averaged accuracy metric achieved by the neural network on the data points of the test set.** You may use a loop to enumerate the different categories, but you will get point deductions for iterating over data points.

**!! Solve this task with the use of only Python, NumPy and/or PyTorch ! Do not use Scikit-learn or other libraries for this task !!**

(Read the next option as well before starting your solution.)

### **Optional task**: additional +3 points

Implement the task described above, but solve it in a fully vectorized way: do not use loops, or similar inefficient constructs, even for enumerating the categories.

---

_**Example calculation of the macro-averaged accuracy metric:**_

We have five data points, which we classify into one of three categories.

Their true labels: [0, 2, 1, 1, 1]

The labels estimated with the highest probability by our neural network: [0, 0, 2, 2, 1]

Then the macro-averaged accuracy metric is computed as follows:
- Category #0: we have one positive data point (belonging to category #0), and it was classified correctly (TP = 1). We have four negative data points (not belonging to category #0), and three of these were correctly classified as negative (TN = 3). We have five data points in total, so the binary accuracy is (1+3)/5 = 4/5.
- Category #1: we have three positive data points (belonging to category #1), and one of these was classified correctly (TP = 1). We have two negative data points (not belonging to category #1), and both were correctly classified as negative (TN = 2). We have five data points in total, so the binary accuracy is (1+2)/5 = 3/5.
- Category #2: we have one positive data points (belonging to category #2), and it was not classified correctly (TP = 0). We have four negative data points (not belonging to category #2), and only two of these were correctly classified as negative (TN = 2). We have five data points in total, so the binary accuracy is (0+2)/5 = 2/5.

The macro-averaged accuracy is the average of the three numbers: (4/5 + 3/5 + 2/5)/3 = 3/5 = 0.6


In [ ]:
# implement your solution BELOW (No automatic tests for this task)







